In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
PROJECT_ROOT = Path.cwd().parents[1]
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

PROJECT_ROOT, DATA_PROCESSED

(PosixPath('/Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction'),
 PosixPath('/Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction/data/processed'))

In [3]:
drhp = pd.read_csv(
    DATA_PROCESSED / "drhp_section_text.csv"
)

drhp.shape

(842, 6)

In [4]:
drhp.columns.tolist()

['company_name', 'year', 'section', 'text', 'pdf_path', 'text_len']

In [9]:
# Load IPO master (RAW → cleaned here)
ipo_master = pd.read_csv(
    DATA_PROCESSED / "ipo_master_stage1.csv"
)

# Standardize columns
ipo_master["listing_date"] = pd.to_datetime(
    ipo_master["DATE OF LISTING"],
    utc=True,
    errors="coerce"
)

ipo_master["company_name"] = (
    ipo_master["COMPANY NAME"]
    .str.lower()
    .str.strip()
)

# Keep only required columns
ipo_master = ipo_master[
    ["company_name", "listing_date"]
].dropna()

ipo_master.shape

(1179, 2)

In [10]:
ipo_master.head()

,company_name,listing_date
0,ruchi soya industries limited,2003-01-02 00:00:00+00:00
1,thejo engineering limited,2012-09-18 00:00:00+00:00
2,thejo engineering limited,2012-09-18 00:00:00+00:00
3,veto switchgears and cables limited,2012-12-13 00:00:00+00:00
4,opal luxury time products limited,2013-04-12 00:00:00+00:00


In [11]:
market = pd.read_csv(
    DATA_PROCESSED / "market_features.csv",
    parse_dates=["listing_date"]
)

market.shape

(1156, 11)

In [13]:
sector = pd.read_csv(
    DATA_PROCESSED / "sector_and_regime_features.csv",
    parse_dates=["listing_date"]
)

sector.shape

(3577, 10)

In [14]:
def normalize_name(x):
    if pd.isna(x):
        return x
    return (
        str(x)
        .lower()
        .replace("&", "and")
        .replace(".", "")
        .replace(",", "")
        .strip()
    )

for df in [drhp, ipo_master, market, sector]:
    if "company_name" in df.columns:
        df["company_name"] = df["company_name"].apply(normalize_name)

In [15]:
from rapidfuzz import process, fuzz

In [16]:
ipo_names = ipo_master["company_name"].dropna().unique().tolist()

In [17]:
def resolve_company_name_safe(text, names, threshold=85):
    if not isinstance(text, str):
        return None
    match = process.extractOne(
        text[:2000],
        names,
        scorer=fuzz.token_set_ratio
    )
    if match and match[1] >= threshold:
        return match[0]
    return None

In [18]:
drhp["resolved_company_name"] = drhp["text"].apply(
    lambda x: resolve_company_name_safe(x, ipo_names, threshold=85)
)

drhp["resolved_company_name"].notna().mean()

np.float64(0.0)

In [21]:
# -------------------------------
# STEP 1: Aggregate DRHP to IPO level
# -------------------------------

drhp_agg = (
    drhp
    .groupby("company_name", as_index=False)
    .agg(
        drhp_total_chars=("text_len", "sum"),
        drhp_sections=("section", "nunique")
    )
)

drhp_agg.shape

KeyError: 'company_name'

In [22]:
drhp.columns.tolist()

['company_name_x',
 'year',
 'section',
 'text',
 'pdf_path',
 'text_len',
 'resolved_company_name',
 'company_name_y',
 'listing_date']

In [23]:
[x for x in drhp.columns if "_x" in x or "_y" in x]

['company_name_x', 'company_name_y']

In [24]:
# Keep only what we truly need
drhp = drhp[
    [
        "resolved_company_name",
        "year",
        "section",
        "text",
        "text_len",
        "pdf_path"
    ]
].copy()

drhp.shape

(0, 6)

In [25]:
drhp = drhp.dropna(subset=["resolved_company_name"]).copy()

drhp.shape

(0, 6)

In [26]:
drhp = drhp.rename(
    columns={"resolved_company_name": "company_name"}
)

drhp.columns.tolist()

['company_name', 'year', 'section', 'text', 'text_len', 'pdf_path']

In [27]:
drhp_agg = (
    drhp
    .groupby("company_name", as_index=False)
    .agg(
        drhp_total_chars=("text_len", "sum"),
        drhp_sections=("section", "nunique")
    )
)

drhp_agg.shape

(0, 3)

In [28]:
df = ipo_master.merge(
    drhp_agg,
    on="company_name",
    how="inner"
)

df.shape

(0, 4)

In [29]:
df[["drhp_total_chars", "drhp_sections"]].describe()

,drhp_total_chars,drhp_sections
count,0.0,0.0
mean,NaN,NaN
std,NaN,NaN
min,NaN,NaN
25%,NaN,NaN
50%,NaN,NaN
75%,NaN,NaN
max,NaN,NaN


In [30]:
import re

def normalize_name(x):
    if pd.isna(x):
        return None
    x = x.lower()
    x = re.sub(r"\(.*?\)", "", x)        # remove brackets
    x = re.sub(r"[^a-z ]", " ", x)       # remove symbols
    x = re.sub(r"\s+", " ", x).strip()   # normalize spaces
    return x

drhp["name_norm"] = drhp["company_name"].apply(normalize_name)
ipo_master["name_norm"] = ipo_master["company_name"].apply(normalize_name)

In [31]:
from rapidfuzz import process, fuzz

ipo_names = ipo_master["name_norm"].dropna().unique()

def match_ipo(name):
    if pd.isna(name):
        return None
    match = process.extractOne(
        name,
        ipo_names,
        scorer=fuzz.token_sort_ratio
    )
    if match and match[1] >= 90:
        return match[0]
    return None

drhp["ipo_name_norm"] = drhp["name_norm"].apply(match_ipo)

drhp["ipo_name_norm"].notna().mean()

nan